In [5]:
from feast import FeatureStore
from datetime import datetime, timedelta, timezone
from typing import List, Dict, Any
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"botocore(\..*)?$",
)
warnings.filterwarnings(
    "ignore",
    message=r"Batch feature views are experimental features in alpha development\..*",
    category=RuntimeWarning,
    module=r"feast\.batch_feature_view$",
)

store = FeatureStore(repo_path=".")

In [ ]:
import pandas as pd
# error:  Spectrum nested query error context: A subquery that refers to a nested table cannot refer to any other table.
# spectrum does not work with feast get_historical_features, must create a local table first!
def fetch_historical_features_entity_sql(
        store: FeatureStore, 
        source: str = "", 
        for_batch_scoring: bool = False, 
        start_date: datetime = None, 
        end_date: datetime = None
):
    from datetime import timedelta

    if end_date is None:
        end_date = (
            datetime.now()
            .replace(microsecond=0, second=0, minute=0)
            .astimezone(tz=timezone.utc)
        )
    if start_date is None:
        start_date = (end_date - timedelta(days=1)).astimezone(tz=timezone.utc)


    source_query = store.get_data_source("criteo_processed").get_table_query_string().replace("\"", "")
    timestamp_col = "GETDATE() as event_timestamp" if for_batch_scoring else "event_timestamp"
    
    entity_sql = f"""
        SELECT
            uid,
            {timestamp_col}
        FROM {source_query}
        WHERE event_timestamp BETWEEN '{start_date.strftime("%Y-%m-%d %H:%M:%S")}' AND '{end_date.strftime("%Y-%m-%d %H:%M:%S")}'
    """
    print(entity_sql)
    if source == "":
        features_to_fetch = [
            "impression_unprocessed_view:campaign",
            "impression_unprocessed_view:cat1",
            "impression_unprocessed_view:cat2",
            "impression_transformed_view:last_5_clicks",
            "impression_transformed_view:last_5_conversions",
        ]
    else:
        features_to_fetch = store.get_feature_service(source)

    return store.get_historical_features(
        features=features_to_fetch,
        entity_df=entity_sql,
    ).to_df()

def fetch_historical_features_entity_df(store: FeatureStore, source: str = "", for_batch_scoring: bool = False, entity_df: pd.DataFrame = None):

    # For batch scoring, we want the latest timestamps
    if for_batch_scoring and entity_df is not None:
        entity_df["event_timestamp"] = pd.to_datetime("now", utc=False)

    if source == "":
        features_to_fetch = [
            "impression_unprocessed_view:campaign",
            "impression_unprocessed_view:cat1",
            "impression_unprocessed_view:cat2",
            "impression_transformed_view:last_5_clicks",
            "impression_transformed_view:last_5_conversions",
        ]
    else:
        features_to_fetch = store.get_feature_service(source)

    return store.get_historical_features(
        entity_df=entity_df,
        features=features_to_fetch
    ).to_df()

def fetch_online_features(store, source: str = "", entity_rows: List[Dict[str, Any]] = []):
    if source == "":
        features_to_fetch = [
            "impression_unprocessed_view:campaign",
            "impression_unprocessed_view:cat1",
            "impression_unprocessed_view:cat2",
            "impression_transformed_view:last_5_clicks",
            "impression_transformed_view:last_5_conversions",
        ]
    else:
        features_to_fetch = store.get_feature_service(source)
    
    return store.get_online_features(
        features=features_to_fetch,
        entity_rows=entity_rows,
    ).to_dict()

Test batch feature retrieval from the offline store. Performs point in time join, which means that for each entity id at timestamp t, latest values of features are joined within the range of t - ttl

In [9]:
# from datetime import datetime, timezone

# # Example: fetch features for March 20–25, 2026
# start = datetime(2026, 3, 20, tzinfo=timezone.utc)
# end = datetime(2026, 3, 25, tzinfo=timezone.utc)

# entity_df = pd.DataFrame.from_dict(
#     {
#         # entity's join key -> entity values
#         "uid": [15827231],
#         # "event_timestamp" (reserved key) -> timestamps
#         "event_timestamp": [
#             datetime(2026, 3, 20, 8, 18, 0)
#         ]
#     }
# )

# df = fetch_historical_features_entity_df(
#     store,
#     source="user_activity",
#     for_batch_scoring=False,
#     entity_df=entity_df
# )
# df.tail()

In [10]:
from datetime import datetime, timezone

# Example: fetch features for March 20–25, 2026
start = datetime(2026, 3, 20, tzinfo=timezone.utc)
end = datetime(2026, 3, 25, tzinfo=timezone.utc)

df = fetch_historical_features_entity_sql(
    store,
    source="user_activity",
    for_batch_scoring=False,
    start_date=start,
    end_date=end,
)
df.tail()


        SELECT
            uid,
            event_timestamp
        FROM public.temp_feast_entities
        WHERE event_timestamp BETWEEN '2026-03-20 00:00:00' AND '2026-03-25 00:00:00'
    


,uid,event_timestamp,campaign,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,last_5_clicks,last_5_conversions
552773,12385931,2026-03-20 06:50:34.084127,6686701,1973606,26597095,13830199,29196072,32440044,1973606,6517920,32440044,29196072,,
552774,12424245,2026-03-20 07:06:14.084127,18443077,30763035,9312274,14455973,29196072,5824237,1973606,15925850,5824233,29196072,,
552775,12433485,2026-03-20 07:09:17.084127,5061834,30763035,9312274,28944165,29196072,26611392,26597096,9312274,29196072,29196072,,
552776,12467498,2026-03-20 09:57:58.084127,19602309,138937,9312274,32457986,29196072,5824239,1973606,7345278,32440044,29196072,,
552777,12469289,2026-03-20 03:09:34.084127,12491647,30763035,9312274,1423185,29196072,32440044,1973606,30116699,29196072,29196072,,


Verify that online feature store only pulls the latest values within the date range. The following was tested against "feast materialize 2026-03-19T00:00:00 2026-03-25T00:00:00"

latest feature values for uid=15827231 in the online store comes from event_timestamp=2026-03-20 03:09:34.084127, which is correct!

In [12]:
fetch_online_features(store, source="user_activity", entity_rows=[{"uid": 15827231}])

{'uid': [15827231],
 'cat6': [5824235],
 'last_5_clicks': ['3960115,6513183,10474106,29427842,30368132'],
 'cat4': [29196072],
 'cat2': [7477605],
 'cat3': [15435275],
 'campaign': [29427842],
 'cat1': [5642940],
 'cat7': [9312274],
 'cat5': [5824237],
 'last_5_conversions': [''],
 'cat8': [29196072],
 'cat9': [358246]}